# 🔎 Etapa 0: Exploração Prévia de Domínios Candidatos

Este notebook ajuda a escolher **com base em dados** quais domínios (categorias) devem ser usados no experimento de *Domain Shift* e *Language Shift*.

A escolha incorreta dos domínios (ex: "Livros") pode causar problemas nas etapas seguintes porque a base em Português (B2W/Olist) pode ter um número irrisório de **avaliações negativas** para aquela categoria, forçando todo o subconjunto de testes (e às vezes de treino) a ter um tamanho muito reduzido devido à regra de balanceamento.

## Objetivo deste script:
Baixar a base do B2W, mapear os rótulos de sentimento e descobrir quais as maiores categorias, focando naquelas que possuem um número mais robusto na classe minoritária (quase sempre a negativa).


In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Download do B2W
CAMINHO_B2W = "/content/b2w.csv"
URL_B2W = "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv"

print("📥 Baixando B2W Reviews...")
if not os.path.exists(CAMINHO_B2W):
    if not os.path.exists("/content"):
        os.makedirs("/content", exist_ok=True)
    df_b2w = pd.read_csv(URL_B2W, low_memory=False)
    df_b2w.to_csv(CAMINHO_B2W, index=False)
else:
    df_b2w = pd.read_csv(CAMINHO_B2W, low_memory=False)
    
print(f"✅ Base B2W carregada com {df_b2w.shape[0]:,} avaliações.")

In [ ]:
# 2. Mapeamento de Rótulos
def estrela_para_sentimento(nota):
    try:
        n = float(nota)
    except (TypeError, ValueError):
        return None
    if n <= 2: return 'Negativo'  # 0 no script final, mas string ajuda na visualizacao aqui
    if n >= 4: return 'Positivo'  # 1 no script final
    return None

df_b2w['label'] = df_b2w['overall_rating'].apply(estrela_para_sentimento)
df_b2w_validos = df_b2w.dropna(subset=['label']).copy()

print(f"🔹 Após descartar notas 3 (neutras) e nulas, sobraram {df_b2w_validos.shape[0]:,} avaliações.")
print(df_b2w_validos['label'].value_counts())

In [ ]:
# 3. Agrupamento por Categoria (site_category_lv1)
# Vamos descobrir quais categorias têm uma boa representação nas DUAS classes.

# Tabela Pivot: Categorias x Sentimentos
tabela_categorias = pd.crosstab(df_b2w_validos['site_category_lv1'], df_b2w_validos['label'])

# O Gargalo é o mínimo entre Positivos e Negativos, 
# já que o modelo precisará fazer undersampling (balanceamento) por classe
tabela_categorias['Gargalo (Balanceado)'] = tabela_categorias.min(axis=1)
tabela_categorias['Total'] = tabela_categorias['Negativo'] + tabela_categorias['Positivo']

# Ordenando pelo gargalo (aquelas que nos permitirão o maior número final equilibrado)
tabela_categorias = tabela_categorias.sort_values(by='Gargalo (Balanceado)', ascending=False)

print("🏆 TOP 15 Categorias mais 'seguras' para serem escolhidas como domínio (ordenadas pelo gargalo):\n")
display(tabela_categorias.head(15))

In [ ]:
# 4. Visualização
plt.figure(figsize=(12, 8))
sns.set_theme(style="whitegrid")

top_10 = tabela_categorias.head(10).reset_index()

# Plotando barras do 'Gargalo (Balanceado)'
ax = sns.barplot(x='Gargalo (Balanceado)', y='site_category_lv1', data=top_10, color='royalblue')
plt.title('Top 10 Categorias por Amostras Balanceadas (Amostras da Classe Minoritária)', fontsize=14)
plt.xlabel('Número máximo de amostras possíveis por classe (Balanceado)', fontsize=12)
plt.ylabel('Categoria (B2W)', fontsize=12)

# Anotando o número nos gráficos
for p in ax.patches:
    ax.annotate(f'{int(p.get_width()):,}', 
                (p.get_width(), p.get_y() + p.get_height() / 2.), 
                ha = 'left', va = 'center', 
                xytext = (5, 0), 
                textcoords = 'offset points')

plt.show()

## 💡 Como interpretar os resultados acima:

- As categorias no topo da lista possuem as maiores margens de segurança para gerar as partições `S3` (Domínio 1 - PT) e `S4` (Domínio 2 - PT).
- Se a categoria escolhida tem um 'Gargalo' de 10.000, isso significa que você conseguirá extrair no máximo 10.000 amostras positivas e 10.000 negativas (20.000 no total) daquela categoria.
- Você deve escolher **duas** categorias do topo que sejam **semanticamente distintas** (por exemplo: Celulares vs Beleza & Saúde). E certificar-se de que a Amazon Reviews americana também possua boas amostras nesses dois domínios.

A partir dessa escolha, você deve mapear as *keywords* associadas na etapa seguinte (Etapa 1).